In [6]:
%pip install ydata_profiling nltk Sastraw

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement Sastraw (from versions: none)
ERROR: No matching distribution found for Sastraw


In [7]:
import pandas as pd

df_train = pd.read_csv('https://raw.githubusercontent.com/IndoNLP/nusax/refs/heads/main/datasets/sentiment/indonesian/train.csv')
df_valid = pd.read_csv('https://raw.githubusercontent.com/IndoNLP/nusax/refs/heads/main/datasets/sentiment/indonesian/valid.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/IndoNLP/nusax/refs/heads/main/datasets/sentiment/indonesian/test.csv')

print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(500, 3)
(100, 3)
(400, 3)


In [8]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_train, title="Profiling Report")

profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 45.90it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
import re
import nltk
from typing import List
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# download stopwords jika belum ada
nltk.download('stopwords')

class IndonesianTextPreprocessor:

    def __init__(self,
                 lowercase: bool = True,
                 remove_special: bool = True,
                 remove_stopwords: bool = True,
                 tokenize: bool = True):

        self.lowercase = lowercase
        self.remove_special = remove_special
        self.remove_stopwords = remove_stopwords
        self.tokenize = tokenize

        # Load stopwords dari dua sumber
        self.stopwords_nltk = set(nltk.corpus.stopwords.words('indonesian'))

        factory = StopWordRemoverFactory()
        self.stopwords_sastrawi = set(factory.get_stop_words())

        # gabungkan
        self.stopwords = self.stopwords_nltk.union(self.stopwords_sastrawi)


    # 1. Lowercase
    def lowercase_text(self, text: str) -> str:
        return text.lower()


    # 2. Remove special characters
    def remove_special_characters(self, text: str) -> str:

        # remove URL
        text = re.sub(r'http\S+|www\S+', '', text)

        # remove mention
        text = re.sub(r'@\w+', '', text)

        # remove hashtag symbol only
        text = re.sub(r'#', '', text)

        # keep only letters and spaces
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        # remove extra spaces
        text = re.sub(r'\s+', ' ', text).strip()

        return text


    # 3. Tokenization
    def tokenize_text(self, text: str) -> List[str]:
        tokens = text.split()
        return tokens


    # 4. Remove stopwords
    def remove_stopwords_from_tokens(self, tokens: List[str]) -> List[str]:
        filtered = [word for word in tokens if word not in self.stopwords]
        return filtered


    # 5. Full pipeline for single text
    def preprocess_text(self, text: str) -> str:

        if self.lowercase:
            text = self.lowercase_text(text)

        if self.remove_special:
            text = self.remove_special_characters(text)

        if self.tokenize:
            tokens = self.tokenize_text(text)
        else:
            tokens = [text]

        if self.remove_stopwords:
            tokens = self.remove_stopwords_from_tokens(tokens)

        cleaned_text = " ".join(tokens)

        return cleaned_text


    # 6. Preprocess dataset (list of text)
    def preprocess_dataset(self, texts: List[str]) -> List[str]:

        cleaned_texts = []

        for text in texts:
            cleaned = self.preprocess_text(text)
            cleaned_texts.append(cleaned)

        return cleaned_texts

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KuePuki\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
preprocessor = IndonesianTextPreprocessor(remove_stopwords=False)

df_train_clean = df_train.copy()
df_valid_clean = df_valid.copy()
df_test_clean = df_test.copy()

df_train_clean['text'] = preprocessor.preprocess_dataset(df_train['text'])
df_valid_clean['text'] = preprocessor.preprocess_dataset(df_valid['text'])
df_test_clean['text'] = preprocessor.preprocess_dataset(df_test['text'])

df_train_clean

,id,text,label
0,219,nikmati cicilan hingga bulan untuk pemesanan t...,neutral
1,209,kuekue yang disajikan bikin saya bernostalgia ...,positive
2,436,ibu pernah bekerja di grab indonesia,neutral
3,394,paling suka banget makan siang di sini ayam sa...,positive
4,592,pelayanan bus damri sangat baik,positive
...,...,...,...
495,589,si a omongnya tong kosong nyaring bunyinya bic...,negative
496,636,sambalnya tidak akan ada di tempat lain rasa t...,positive
497,710,menurut saya steaknya cukup enak hanya lebih b...,positive
498,250,dijaga ya makannya gus emang lagi musimnya sek...,negative


In [11]:
%pip install scikit-learn sentence-transformers transformers torch gensim

Note: you may need to restart the kernel to use updated packages.


In [12]:
from abc import ABC, abstractmethod
import numpy as np

class BaseEmbedding(ABC):

    @abstractmethod
    def fit(self, texts):
        pass

    @abstractmethod
    def transform(self, texts):
        pass

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEmbedding(BaseEmbedding):

    def __init__(self, max_features=10000):
        self.vectorizer = TfidfVectorizer(max_features=max_features)

    def fit(self, texts):
        self.vectorizer.fit(texts)

    def transform(self, texts):
        return self.vectorizer.transform(texts).toarray()

In [14]:
from sentence_transformers import SentenceTransformer

class SBERTEmbedding(BaseEmbedding):

    def __init__(self, model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.model = SentenceTransformer(model_name)

    def fit(self, texts):
        pass  # SBERT tidak perlu training

    def transform(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

In [15]:
import gensim.downloader as api
import numpy as np

class FastTextEmbedding(BaseEmbedding):

    def __init__(self, embedding_dim=300, max_length=50):

        self.model = api.load("fasttext-wiki-news-subwords-300")
        self.embedding_dim = embedding_dim
        self.max_length = max_length

    def fit(self, texts):
        pass

    def text_to_sequence(self, text):

        words = text.split()
        sequence = []

        for word in words[:self.max_length]:

            if word in self.model:
                sequence.append(self.model[word])
            else:
                sequence.append(np.zeros(self.embedding_dim))

        while len(sequence) < self.max_length:
            sequence.append(np.zeros(self.embedding_dim))

        return np.array(sequence)

    def transform(self, texts):

        sequences = []

        for text in texts:
            seq = self.text_to_sequence(text)
            sequences.append(seq)

        return np.array(sequences)

In [16]:
import torch
from transformers import AutoTokenizer, AutoModel

class BERTEmbedding(BaseEmbedding):

    def __init__(self, model_name, max_length=50, device=None):

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

        self.max_length = max_length

        if device:
            self.device = device
        else:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model.to(self.device)
        self.model.eval()


    def fit(self, texts):
        pass


    def transform(self, texts):

        embeddings = []

        with torch.no_grad():

            for text in texts:

                inputs = self.tokenizer(
                    text,
                    padding='max_length',
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt"
                )

                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                outputs = self.model(**inputs)

                hidden_states = outputs.last_hidden_state

                embeddings.append(hidden_states.squeeze().cpu().numpy())

        return np.array(embeddings)

In [17]:
class EmbeddingFactory:

    @staticmethod
    def create(embedding_type):

        if embedding_type == "tfidf":
            return TfidfEmbedding()

        elif embedding_type == "sbert":
            return SBERTEmbedding()

        elif embedding_type == "fasttext":
            return FastTextEmbedding()

        elif embedding_type == "indobert":
            return BERTEmbedding("indobenchmark/indobert-base-p1")

        elif embedding_type == "mbert":
            return BERTEmbedding("bert-base-multilingual-cased")

        else:
            raise ValueError("Unknown embedding type")

In [18]:
tfidf_embedding = EmbeddingFactory.create("tfidf")
tfidf_embed = tfidf_embedding.fit_transform(df_train_clean['text'])
print(tfidf_embed.shape)

sbert_embedding = EmbeddingFactory.create("sbert")
sbert_embed = sbert_embedding.fit_transform(df_train_clean['text'])
print(sbert_embed.shape)

fasttext_embedding = EmbeddingFactory.create("fasttext")
fasttext_embed = fasttext_embedding.fit_transform(df_train_clean['text'])
print(fasttext_embed.shape)

indobert_embedding = EmbeddingFactory.create("indobert")
indobert_embed = indobert_embedding.fit_transform(df_train_clean['text'])
print(indobert_embed.shape)

mbert_embedding = EmbeddingFactory.create("mbert")
mbert_embed = mbert_embedding.fit_transform(df_train_clean['text'])
print(mbert_embed.shape)

(500, 2771)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(500, 384)
(500, 50, 300)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(500, 50, 768)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(500, 50, 768)


In [19]:
%pip install matplotlib seaborn scikit-learn umap-learn

Note: you may need to restart the kernel to use updated packages.


In [20]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap


class EmbeddingComparisonVisualizer:

    def __init__(
        self,
        reduction_methods=("pca", "tsne", "umap"),
        random_state=42,
        perplexity=30
    ):

        self.reduction_methods = reduction_methods
        self.random_state = random_state
        self.perplexity = perplexity


    # Mean pooling untuk sequence embedding (BERT, FastText)
    def mean_pool(self, embeddings):

        if len(embeddings.shape) == 3:
            return np.mean(embeddings, axis=1)

        return embeddings


    # Reduce dimensionality
    def reduce(self, embeddings, method):

        if method == "pca":

            reducer = PCA(
                n_components=2,
                random_state=self.random_state
            )

        elif method == "tsne":

            reducer = TSNE(
                n_components=2,
                random_state=self.random_state,
                perplexity=self.perplexity
            )

        elif method == "umap":

            reducer = umap.UMAP(
                n_components=2,
                random_state=self.random_state
            )

        else:
            raise ValueError("Unknown reduction method")

        return reducer.fit_transform(embeddings)


    # Plot grid untuk multiple embeddings dan multiple reduction methods
    def plot_grid(self, embeddings_dict, labels):

        """
        embeddings_dict:
        {
            "TF-IDF": embeddings,
            "SBERT": embeddings,
            "IndoBERT": embeddings
        }
        """

        n_embeddings = len(embeddings_dict)
        n_methods = len(self.reduction_methods)

        fig, axes = plt.subplots(
            n_embeddings,
            n_methods,
            figsize=(6*n_methods, 5*n_embeddings)
        )

        # Jika hanya satu row
        if n_embeddings == 1:
            axes = [axes]

        for row_idx, (embed_name, embeddings) in enumerate(embeddings_dict.items()):

            # mean pool jika sequence embedding
            embeddings = self.mean_pool(embeddings)

            for col_idx, method in enumerate(self.reduction_methods):

                reduced = self.reduce(embeddings, method)

                ax = axes[row_idx][col_idx] if n_embeddings > 1 else axes[col_idx]

                sns.scatterplot(
                    x=reduced[:, 0],
                    y=reduced[:, 1],
                    hue=labels,
                    palette="tab10",
                    s=40,
                    ax=ax,
                    legend=(col_idx == n_methods-1)
                )

                ax.set_title(f"{embed_name} + {method.upper()}")
                ax.set_xlabel("Component 1")
                ax.set_ylabel("Component 2")

        plt.tight_layout()
        plt.show()


In [21]:
embeddings_dict = {
    "TF-IDF": tfidf_embed,
    "SBERT": sbert_embed,
    "FastText": fasttext_embed,
    "IndoBERT": indobert_embed,
    "mBERT": mbert_embed
}

visualizer = EmbeddingComparisonVisualizer()

visualizer.plot_grid(embeddings_dict, df_train_clean['label'])

# Modeling & Testing

We will train 4 classifiers on 5 embedding types:
1. **Logistic Regression**
2. **SVM**
3. **Random Forest**
4. **K-Nearest Neighbors**

Evaluation metrics:
- Accuracy
- Weighted F1-Score

In [22]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(df_train_clean['label'])
y_valid = le.transform(df_valid_clean['label'])
y_test = le.transform(df_test_clean['label'])

print("Classes:", le.classes_)
print("Train shape:", y_train.shape)

Classes: ['negative' 'neutral' 'positive']
Train shape: (500,)


In [23]:
# Transform Validation & Test Sets with all embeddings

# TF-IDF
tfidf_valid = tfidf_embedding.transform(df_valid_clean['text'])
tfidf_test = tfidf_embedding.transform(df_test_clean['text'])

# SBERT
sbert_valid = sbert_embedding.transform(df_valid_clean['text'])
sbert_test = sbert_embedding.transform(df_test_clean['text'])

# FastText
fasttext_valid = fasttext_embedding.transform(df_valid_clean['text'])
fasttext_test = fasttext_embedding.transform(df_test_clean['text'])

# IndoBERT
indobert_valid = indobert_embedding.transform(df_valid_clean['text'])
indobert_test = indobert_embedding.transform(df_test_clean['text'])

# mBERT
mbert_valid = mbert_embedding.transform(df_valid_clean['text'])
mbert_test = mbert_embedding.transform(df_test_clean['text'])

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [24]:
# Helper to mean pool 3D embeddings (FastText, BERT) -> 2D
def get_pooled_embeddings(embedding_name, train, valid, test):
    
    # If already 2D, return as is
    if len(train.shape) == 2:
        return train, valid, test
        
    # Mean pool over sequence length (axis 1)
    print(f"Pooling {embedding_name} from {train.shape}...")
    return np.mean(train, axis=1), np.mean(valid, axis=1), np.mean(test, axis=1)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier()
}

# Define embedding dictionary
embedding_data = {
    "TF-IDF": (tfidf_embed, tfidf_valid, tfidf_test),
    "SBERT": (sbert_embed, sbert_valid, sbert_test),
    "FastText": (fasttext_embed, fasttext_valid, fasttext_test),
    "IndoBERT": (indobert_embed, indobert_valid, indobert_test),
    "mBERT": (mbert_embed, mbert_valid, mbert_test)
}

results = []

for embed_name, (X_train_raw, X_valid_raw, X_test_raw) in embedding_data.items():
    
    # Pool if needed
    X_train, X_valid, X_test = get_pooled_embeddings(embed_name, X_train_raw, X_valid_raw, X_test_raw)
    
    print(f"\nEvaluating {embed_name} (shape: {X_train.shape})...")
    
    for clf_name, clf in classifiers.items():
        
        # Train
        clf.fit(X_train, y_train)
        
        # Predict on Test
        y_pred = clf.predict(X_test)
        
        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results.append({
            "Embedding": embed_name,
            "Classifier": clf_name,
            "Accuracy": acc,
            "F1-Score": f1
        })
        
        print(f"  {clf_name}: Accuracy={acc:.4f}, F1={f1:.4f}")


Evaluating TF-IDF (shape: (500, 2771))...
  Logistic Regression: Accuracy=0.7600, F1=0.7509
  SVM: Accuracy=0.7250, F1=0.7004
  Random Forest: Accuracy=0.7350, F1=0.7322
  KNN: Accuracy=0.6725, F1=0.6557

Evaluating SBERT (shape: (500, 384))...
  Logistic Regression: Accuracy=0.8375, F1=0.8347
  SVM: Accuracy=0.8625, F1=0.8619
  Random Forest: Accuracy=0.8050, F1=0.8012
  KNN: Accuracy=0.7575, F1=0.7461
Pooling FastText from (500, 50, 300)...

Evaluating FastText (shape: (500, 300))...
  Logistic Regression: Accuracy=0.5450, F1=0.5237
  SVM: Accuracy=0.5925, F1=0.5899
  Random Forest: Accuracy=0.6125, F1=0.6061
  KNN: Accuracy=0.5525, F1=0.5393
Pooling IndoBERT from (500, 50, 768)...

Evaluating IndoBERT (shape: (500, 768))...
  Logistic Regression: Accuracy=0.8925, F1=0.8925
  SVM: Accuracy=0.9000, F1=0.8992
  Random Forest: Accuracy=0.8800, F1=0.8781
  KNN: Accuracy=0.8350, F1=0.8299
Pooling mBERT from (500, 50, 768)...

Evaluating mBERT (shape: (500, 768))...
  Logistic Regression:

In [26]:
# Display Results Table
df_results = pd.DataFrame(results)
df_results_sorted = df_results.sort_values(by="F1-Score", ascending=False)
df_results_sorted

,Embedding,Classifier,Accuracy,F1-Score
13,IndoBERT,SVM,0.9000,0.899165
12,IndoBERT,Logistic Regression,0.8925,0.892503
14,IndoBERT,Random Forest,0.8800,0.878118
5,SBERT,SVM,0.8625,0.861871
4,SBERT,Logistic Regression,0.8375,0.834670
15,IndoBERT,KNN,0.8350,0.829946
6,SBERT,Random Forest,0.8050,0.801193
17,mBERT,SVM,0.7650,0.763217
0,TF-IDF,Logistic Regression,0.7600,0.750872
7,SBERT,KNN,0.7575,0.746141


In [27]:
# Plot Comparison Heatmap
pivot_acc = df_results.pivot(index="Embedding", columns="Classifier", values="Accuracy")
pivot_f1 = df_results.pivot(index="Embedding", columns="Classifier", values="F1-Score")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pivot_acc, annot=True, cmap="YlGnBu", fmt=".3f", ax=axes[0])
axes[0].set_title("Accuracy Comparison")

sns.heatmap(pivot_f1, annot=True, cmap="YlGnBu", fmt=".3f", ax=axes[1])
axes[1].set_title("Weighted F1-Score Comparison")

plt.tight_layout()
plt.show()

In [28]:
# Confusion Matrix for Best Model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_row = df_results_sorted.iloc[0]
best_embed_name = best_row["Embedding"]
best_clf_name = best_row["Classifier"]

print(f"Best Model: {best_clf_name} with {best_embed_name}")

# Retrain best model to get confusion matrix
X_train_raw, _, X_test_raw = embedding_data[best_embed_name]
X_train, _, X_test = get_pooled_embeddings(best_embed_name, X_train_raw, _, X_test_raw)

clf = classifiers[best_clf_name]
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix: {best_clf_name} + {best_embed_name}")
plt.show()

Best Model: SVM with IndoBERT
Pooling IndoBERT from (500, 50, 768)...


---
## LSTM Sentiment Analysis
### Menggunakan Berbagai Vektor Embedding: TF-IDF, SBERT, FastText, IndoBERT, mBERT
Model BiLSTM dibangun dengan TensorFlow/Keras. Input vektor embedding di-reshape menjadi sekuen panjang-1 agar sesuai input LSTM.

In [29]:
# ============================================================
# CELL 1: Install Dependensi
# ============================================================
%pip install -q tensorflow scikit-learn gensim sentence-transformers transformers torch

Note: you may need to restart the kernel to use updated packages.


In [30]:
# ============================================================
# CELL 2: Import & Konfigurasi
# ============================================================
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, Input, Reshape
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('TensorFlow:', tf.__version__)
print('GPU tersedia:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU tersedia: []


In [31]:
# ============================================================
# CELL 3: Fungsi Ekstraksi Embedding
# ============================================================

# --- TF-IDF ---
def get_tfidf_vectors(df_train, df_valid, df_test, text_col='text', max_features=5000):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(max_features=max_features)
    X_tr = vec.fit_transform(df_train[text_col]).toarray()
    X_vl = vec.transform(df_valid[text_col]).toarray()
    X_te = vec.transform(df_test[text_col]).toarray()
    print(f'[TF-IDF] dim={X_tr.shape[1]}')
    return X_tr, X_vl, X_te

# --- SBERT ---
def get_sbert_vectors(df_train, df_valid, df_test, text_col='text',
                      model_name='firqaaa/indo-sentence-bert-base'):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    X_tr = model.encode(df_train[text_col].tolist(), show_progress_bar=True, batch_size=32)
    X_vl = model.encode(df_valid[text_col].tolist(), show_progress_bar=True, batch_size=32)
    X_te = model.encode(df_test[text_col].tolist(),  show_progress_bar=True, batch_size=32)
    print(f'[SBERT] dim={X_tr.shape[1]}')
    return X_tr, X_vl, X_te

# --- FastText (latih dari data jika tidak ada pre-trained) ---
def get_fasttext_vectors(df_train, df_valid, df_test, text_col='text',
                         vector_size=300, fasttext_model_path=None):
    from gensim.models.fasttext import FastText
    def mean_vec(texts, mdl, size):
        out = []
        for t in texts:
            ws  = str(t).lower().split()
            wvs = [mdl.wv[w] for w in ws if w in mdl.wv]
            out.append(np.mean(wvs, axis=0) if wvs else np.zeros(size))
        return np.array(out)
    if fasttext_model_path and os.path.exists(fasttext_model_path):
        from gensim.models.fasttext import load_facebook_vectors
        mdl = load_facebook_vectors(fasttext_model_path)
    else:
        sents = [str(t).lower().split() for t in df_train[text_col]]
        mdl   = FastText(sentences=sents, vector_size=vector_size,
                         window=5, min_count=1, epochs=10, workers=4)
    X_tr = mean_vec(df_train[text_col].tolist(), mdl, vector_size)
    X_vl = mean_vec(df_valid[text_col].tolist(), mdl, vector_size)
    X_te = mean_vec(df_test[text_col].tolist(),  mdl, vector_size)
    print(f'[FastText] dim={X_tr.shape[1]}')
    return X_tr, X_vl, X_te

# --- IndoBERT / mBERT (CLS pooling) ---
def get_bert_vectors(df_train, df_valid, df_test, model_name, text_col='text',
                     batch_size=16, max_length=128):
    import torch
    from transformers import AutoTokenizer, AutoModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'[{model_name.split("/")[-1]}] device={device}')
    tok  = AutoTokenizer.from_pretrained(model_name)
    bert = AutoModel.from_pretrained(model_name).eval().to(device)
    def encode(texts):
        vecs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc   = tok(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors='pt')
            enc   = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad():
                out = bert(**enc)
            vecs.append(out.last_hidden_state[:, 0].cpu().numpy())
        return np.vstack(vecs)
    X_tr = encode(df_train[text_col].tolist())
    X_vl = encode(df_valid[text_col].tolist())
    X_te = encode(df_test[text_col].tolist())
    print(f'  dim={X_tr.shape[1]}')
    return X_tr, X_vl, X_te

print('Fungsi embedding siap.')

Fungsi embedding siap.


In [32]:
# ============================================================
# CELL 4: Arsitektur Model BiLSTM
# ============================================================

def build_lstm_model(input_dim, num_classes, lstm_units=128, dropout=0.3):
    """
    Input shape : (batch, input_dim) — vektor embedding
    Reshape     : (batch, 1, input_dim) — sekuen panjang 1
    BiLSTM      : menangkap pola forward & backward
    """
    inp  = Input(shape=(input_dim,), name='embedding_input')
    x    = Reshape((1, input_dim))(inp)
    x    = Bidirectional(
               LSTM(lstm_units, dropout=dropout, recurrent_dropout=dropout/2)
           )(x)
    x    = Dropout(dropout)(x)
    x    = Dense(64, activation='relu')(x)
    x    = Dropout(dropout/2)(x)
    out  = Dense(num_classes, activation='softmax')(x)
    mdl  = Model(inp, out)
    mdl.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return mdl

# Preview arsitektur
build_lstm_model(768, 3).summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_input (InputLayer)    │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 1, 768)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       918,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 935,171 (3.57 MB)

 Trainable params: 935,171 (3.57 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# ============================================================
# CELL 5: Training & Evaluasi Semua Embedding
# ============================================================

# Encode label
le_lstm      = LabelEncoder()
y_train_lstm = le_lstm.fit_transform(df_train['label'])
y_valid_lstm = le_lstm.transform(df_valid['label'])
y_test_lstm  = le_lstm.transform(df_test['label'])
NUM_CLASSES  = len(le_lstm.classes_)
print('Kelas:', le_lstm.classes_, '| N kelas:', NUM_CLASSES)

lstm_results   = {}   # {name: {'accuracy', 'y_pred', 'model'}}
lstm_histories = {}   # {name: history.history}
EPOCHS, BATCH  = 30, 32

def run_lstm(name, X_tr, X_vl, X_te):
    print(f'\n>>> BiLSTM + {name}  (input_dim={X_tr.shape[1]})')
    model = build_lstm_model(X_tr.shape[1], NUM_CLASSES)
    cb    = [
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=0)
    ]
    hist  = model.fit(
        X_tr, y_train_lstm,
        validation_data=(X_vl, y_valid_lstm),
        epochs=EPOCHS, batch_size=BATCH,
        callbacks=cb, verbose=1
    )
    y_pred = np.argmax(model.predict(X_te), axis=1)
    acc    = accuracy_score(y_test_lstm, y_pred)
    print(f'  Accuracy: {acc:.4f}')
    print(classification_report(y_test_lstm, y_pred,
                                 target_names=le_lstm.classes_, digits=4))
    lstm_results[name]   = {'accuracy': acc, 'y_pred': y_pred, 'model': model}
    lstm_histories[name] = hist.history

# [1] TF-IDF
X_tr_tf, X_vl_tf, X_te_tf = get_tfidf_vectors(df_train, df_valid, df_test)
run_lstm('TF-IDF', X_tr_tf, X_vl_tf, X_te_tf)

# [2] SBERT
X_tr_sb, X_vl_sb, X_te_sb = get_sbert_vectors(df_train, df_valid, df_test)
run_lstm('SBERT', X_tr_sb, X_vl_sb, X_te_sb)

# [3] FastText
X_tr_ft, X_vl_ft, X_te_ft = get_fasttext_vectors(df_train, df_valid, df_test)
run_lstm('FastText', X_tr_ft, X_vl_ft, X_te_ft)

# [4] IndoBERT
X_tr_ib, X_vl_ib, X_te_ib = get_bert_vectors(
    df_train, df_valid, df_test, 'indobenchmark/indobert-base-p1')
run_lstm('IndoBERT', X_tr_ib, X_vl_ib, X_te_ib)

# [5] mBERT
X_tr_mb, X_vl_mb, X_te_mb = get_bert_vectors(
    df_train, df_valid, df_test, 'bert-base-multilingual-cased')
run_lstm('mBERT', X_tr_mb, X_vl_mb, X_te_mb)

# Ringkasan
print('\n' + '='*52)
print('  RINGKASAN TEST ACCURACY — BiLSTM')
print('='*52)
for n, r in lstm_results.items():
    print(f'  {n:<12}: {r["accuracy"]:.4f}')
print('='*52)

Kelas: ['negative' 'neutral' 'positive'] | N kelas: 3
[TF-IDF] dim=2777

>>> BiLSTM + TF-IDF  (input_dim=2777)
Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.4220 - loss: 1.0918 - val_accuracy: 0.6200 - val_loss: 1.0764 - learning_rate: 0.0010
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6780 - loss: 1.0464 - val_accuracy: 0.6700 - val_loss: 1.0270 - learning_rate: 0.0010
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7060 - loss: 0.9429 - val_accuracy: 0.6800 - val_loss: 0.9255 - learning_rate: 0.0010
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7820 - loss: 0.7403 - val_accuracy: 0.7200 - val_loss: 0.7728 - learning_rate: 0.0010
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8880 - loss: 0.4914 - val_accuracy: 0.7600 - val_loss: 0.6419 - learning_rate: 0.0010
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9600 - loss: 0.2832 - val_accuracy: 0.7900 - val_loss: 0.5581 - learn

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  dim=768

>>> BiLSTM + IndoBERT  (input_dim=768)
Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.5140 - loss: 0.9800 - val_accuracy: 0.7500 - val_loss: 0.7699 - learning_rate: 0.0010
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6560 - loss: 0.7614 - val_accuracy: 0.7600 - val_loss: 0.6291 - learning_rate: 0.0010
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7640 - loss: 0.5964 - val_accuracy: 0.8100 - val_loss: 0.5652 - learning_rate: 0.0010
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8060 - loss: 0.5364 - val_accuracy: 0.8200 - val_loss: 0.4988 - learning_rate: 0.0010
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8080 - loss: 0.4904 - val_accuracy: 0.8400 - val_loss: 0.4977 - learning_rate: 0.0010
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8260 - loss: 0.4286 - val_accuracy: 0.8000 - val_loss: 0.5487 - learning_rate: 0.0010
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  dim=768

>>> BiLSTM + mBERT  (input_dim=768)
Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.4460 - loss: 1.0542 - val_accuracy: 0.6000 - val_loss: 0.9710 - learning_rate: 0.0010
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6060 - loss: 0.9177 - val_accuracy: 0.6000 - val_loss: 0.8352 - learning_rate: 0.0010
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6400 - loss: 0.8117 - val_accuracy: 0.6400 - val_loss: 0.8098 - learning_rate: 0.0010
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6780 - loss: 0.7388 - val_accuracy: 0.7000 - val_loss: 0.6992 - learning_rate: 0.0010
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7020 - loss: 0.6848 - val_accuracy: 0.7300 - val_loss: 0.6648 - learning_rate: 0.0010
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7480 - loss: 0.6296 - val_accuracy: 0.7000 - val_loss: 0.7213 - learning_rate: 0.0010
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - 

In [34]:
# ============================================================
# CELL 6: Plot — Training History (Loss & Accuracy)
# ============================================================
EMBED_NAMES = list(lstm_histories.keys())
N      = len(EMBED_NAMES)
COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig, axes = plt.subplots(N, 2, figsize=(14, 4 * N))
fig.suptitle('BiLSTM — Training History per Embedding',
             fontsize=16, fontweight='bold', y=1.01)

for i, name in enumerate(EMBED_NAMES):
    h  = lstm_histories[name]
    c  = COLORS[i]
    al = axes[i, 0]
    ar = axes[i, 1]

    al.plot(h['loss'],         color=c, lw=2, label='Train')
    al.plot(h['val_loss'],     color=c, lw=2, ls='--', alpha=.7, label='Val')
    al.set_title(f'{name} — Loss', fontsize=11)
    al.set_xlabel('Epoch'); al.set_ylabel('Loss')
    al.legend(); al.grid(alpha=.3)

    ar.plot(h['accuracy'],     color=c, lw=2, label='Train')
    ar.plot(h['val_accuracy'], color=c, lw=2, ls='--', alpha=.7, label='Val')
    ar.set_title(f'{name} — Accuracy', fontsize=11)
    ar.set_xlabel('Epoch'); ar.set_ylabel('Accuracy')
    ar.set_ylim(0, 1); ar.legend(); ar.grid(alpha=.3)

plt.tight_layout()
plt.savefig('lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: lstm_training_history.png')

Disimpan: lstm_training_history.png


In [35]:
# ============================================================
# CELL 7: Plot — Bar Accuracy & Grouped F1 per Kelas
# ============================================================
EMBED_NAMES = list(lstm_results.keys())
classes_    = le_lstm.classes_
accs        = [lstm_results[n]['accuracy'] for n in EMBED_NAMES]

# F1 per kelas
f1_matrix = np.zeros((len(EMBED_NAMES), len(classes_)))
for i, name in enumerate(EMBED_NAMES):
    f1_matrix[i] = f1_score(y_test_lstm, lstm_results[name]['y_pred'], average=None)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('BiLSTM — Perbandingan Performa Embedding',
             fontsize=14, fontweight='bold')

# Kiri: Accuracy bar
ax = axes[0]
bars = ax.bar(EMBED_NAMES, accs, color=COLORS[:len(EMBED_NAMES)],
              edgecolor='white', linewidth=1.2, zorder=3)
ax.set_ylim(0, 1.1); ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Test Accuracy', fontsize=13)
ax.grid(axis='y', alpha=.35, zorder=0)
ax.spines[['top','right']].set_visible(False)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + .012,
            f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold')

# Kanan: F1 per kelas
ax2 = axes[1]
x   = np.arange(len(classes_))
w   = 0.15
off = np.linspace(-(len(EMBED_NAMES)-1)*w/2,
                   (len(EMBED_NAMES)-1)*w/2, len(EMBED_NAMES))
for i, (name, o) in enumerate(zip(EMBED_NAMES, off)):
    ax2.bar(x + o, f1_matrix[i], width=w, label=name,
            color=COLORS[i], edgecolor='white', zorder=3)
ax2.set_xticks(x); ax2.set_xticklabels(classes_, fontsize=11)
ax2.set_ylim(0, 1.1); ax2.set_ylabel('F1-Score', fontsize=12)
ax2.set_title('F1-Score per Kelas', fontsize=13)
ax2.legend(fontsize=9, loc='lower right')
ax2.grid(axis='y', alpha=.35, zorder=0)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('lstm_accuracy_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Disimpan: lstm_accuracy_f1.png')

Disimpan: lstm_accuracy_f1.png


In [36]:
# ============================================================
# CELL 8: Plot — Heatmap Metrik & Confusion Matrix Model Terbaik
# ============================================================
EMBED_NAMES = list(lstm_results.keys())

# Buat tabel metrik
rows = []
for name in EMBED_NAMES:
    yp  = lstm_results[name]['y_pred']
    rows.append({
        'Embedding' : name,
        'Accuracy'  : accuracy_score(y_test_lstm, yp),
        'Precision' : precision_score(y_test_lstm, yp, average='weighted'),
        'Recall'    : recall_score(y_test_lstm, yp, average='weighted'),
        'F1 (w-avg)': f1_score(y_test_lstm, yp, average='weighted'),
    })
df_metrics = pd.DataFrame(rows).set_index('Embedding')
print(df_metrics.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('BiLSTM — Ringkasan Metrik & Confusion Matrix Terbaik',
             fontsize=14, fontweight='bold')

# Kiri: Heatmap metrik
sns.heatmap(df_metrics, annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=.5, linecolor='white', ax=axes[0],
            cbar_kws={'shrink': .8}, vmin=0, vmax=1)
axes[0].set_title('Heatmap Metrik Evaluasi', fontsize=12)
axes[0].tick_params(axis='x', rotation=30)

# Kanan: Confusion Matrix model terbaik
best_name   = max(lstm_results, key=lambda n: lstm_results[n]['accuracy'])
cm          = confusion_matrix(y_test_lstm, lstm_results[best_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_lstm.classes_,
            yticklabels=le_lstm.classes_,
            linewidths=.5, linecolor='white', ax=axes[1],
            cbar_kws={'shrink': .8})
axes[1].set_title(f'Confusion Matrix — {best_name} (Best)', fontsize=12)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('lstm_heatmap_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Disimpan: lstm_heatmap_cm.png')
print(f'\n✅ Model Terbaik: BiLSTM + {best_name}',
      f'| Accuracy: {lstm_results[best_name]["accuracy"]:.4f}')

           Accuracy  Precision  Recall  F1 (w-avg)
Embedding                                         
TF-IDF       0.7725     0.7783  0.7725      0.7660
SBERT        0.8575     0.8594  0.8575      0.8554
FastText     0.4925     0.3681  0.4925      0.4153
IndoBERT     0.8575     0.8587  0.8575      0.8549
mBERT        0.7425     0.7421  0.7425      0.7406
Disimpan: lstm_heatmap_cm.png

✅ Model Terbaik: BiLSTM + SBERT | Accuracy: 0.8575
